In [18]:
import modified_didppy as m_dp
import vrplib
import numpy as np
import math
import tempfile
import os
from scipy.spatial.distance import cdist
import pulp
import re

# **Data**

## A dataset

### Load the data

In [8]:
# 1. Define your directory path
# TIP: Use r"..." (raw string) so Python treats backslashes as text, not escape characters
base_path = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_CVRP_dual_bounds_and_models\A"

# 2. Construct the full file paths
# We assume the solution file has the standard .sol extension
instance_path = os.path.join(base_path, "A-n53-k7.vrp")
solution_path = os.path.join(base_path, "A-n53-k7.sol")

# 3. Load the data
try:
    # Read the instance data
    instance = vrplib.read_instance(instance_path)
    
    # Read the solution data
    solution = vrplib.read_solution(solution_path)

    # 4. Print results to verify
    print(f"Successfully loaded: {instance['name']}")
    print(f"Dimension: {instance['dimension']}")
    print(f"Vehicle Capacity: {instance['capacity']}")
    print("-" * 20)
    print(f"Optimal Cost (from solution): {solution['cost']}")
    print(f"Routes: {solution['routes']}")

except FileNotFoundError:
    print("Error: Could not find the file. Please check if 'A-n53-k7.sol' exists in that folder.")

Successfully loaded: A-n53-k7
Dimension: 53
Vehicle Capacity: 100
--------------------
Optimal Cost (from solution): 1010
Routes: [[9, 16, 32, 15, 19, 23, 43, 50, 36, 2, 37], [4, 28, 17, 41, 11, 24, 52, 34], [47, 7, 12, 48, 42, 45, 22], [51, 46, 8, 35, 27], [33, 6, 20, 31], [39, 3, 5, 14, 13, 21, 25], [1, 30, 44, 29, 49, 10, 26, 40, 18, 38]]


In [13]:
print(instance)

{'name': 'A-n53-k7', 'comment': '(Augerat et al, No of trucks: 7, Optimal value: 1010)', 'type': 'CVRP', 'dimension': 53, 'edge_weight_type': 'EUC_2D', 'capacity': 100, 'node_coord': array([[24, 63],
       [35, 60],
       [79, 46],
       [ 3, 45],
       [42, 50],
       [ 3, 40],
       [29, 96],
       [47, 30],
       [54, 77],
       [36, 30],
       [83, 86],
       [30,  6],
       [55, 29],
       [13,  2],
       [ 1, 19],
       [98,  1],
       [75, 10],
       [39, 23],
       [62, 91],
       [96,  9],
       [27, 87],
       [14, 16],
       [52, 49],
       [95, 21],
       [30,  6],
       [18, 40],
       [82, 90],
       [50, 79],
       [48, 49],
       [82, 73],
       [64, 62],
       [34, 78],
       [83,  6],
       [ 3, 77],
       [18,  8],
       [53, 86],
       [88, 51],
       [77, 51],
       [58, 89],
       [12, 44],
       [70, 88],
       [36, 17],
       [85, 23],
       [93, 30],
       [68, 67],
       [71, 34],
       [56, 73],
       [37, 37],
 

### Extract data

In [22]:
# 'instance' is the dictionary you provided in the prompt
# 1. Extract Constraints
capacity = instance['capacity'] # 100
n_nodes = instance['dimension'] # 53
match = re.search(r"No of trucks:\s*(\d+)", instance['comment'])
if match:
    num_vehicles = int(match.group(1))
else:
    print("Number of trucks not found.")
    
# 2. Extract Demands
# Note: instance['demand'] includes the Depot at index 0 (value 0)
# This is usually what you want for 0-based indexing in state representation
demands = instance['demand'] 

# 3. Extract and FIX the Distance Matrix
# The library calculated exact Euclidean distances (floats). 
# For the 'A' (Augerat) series, we typically round to the nearest integer.
dist_matrix = instance['edge_weight']


# --- VERIFICATION ---
print(f"Capacity: {capacity}")
print(f"Number of Nodes: {n_nodes}")
print(f"Number of Vehicles: {num_vehicles}")
print(f"Depot Demand: {demands[0]}")
print(f"Customer 1 Demand: {demands[1]}")

print("\nComparison of Distance (Depot -> Node 1):")
print(f"Distance matrix: {dist_matrix[0][1]}") 

Capacity: 100
Number of Nodes: 53
Number of Vehicles: 7
Depot Demand: 0
Customer 1 Demand: 2

Comparison of Distance (Depot -> Node 1):
Distance matrix: 11.40175425099138


# **CVRP model**

In [29]:
# Sets
n_nodes = n_nodes      # n (Total nodes including depot)
n_customers = n_nodes - 1             # Customers only
n_vehicles = num_vehicles                  # p (As per filename k7)

# Indices
# Nodes: 0 is depot, 1..52 are customers
nodes = range(n_nodes)                # {0, ..., n}
customers = range(1, n_nodes)         # {1, ..., n}
vehicles = range(n_vehicles)          # {0, ..., p-1} (Python 0-indexed)
c = dist_matrix                  # Cost matrix

#Model declaration
model = pulp.LpProblem("CVRP_LP_Relaxation", pulp.LpMinimize)

# --- VARIABLES ---

# X: Flow variables (Relaxed to Continuous 0 to 1)
x = pulp.LpVariable.dicts(
    "x", 
    ((r, i, j) for r in vehicles for i in nodes for j in nodes if i != j), 
    cat='Continuous', lowBound=0, upBound=1
)

# U: MTZ Load variables (MISSING IN YOUR SNIPPET)
# Even in relaxation, these help tighten the bound
u = pulp.LpVariable.dicts(
    "u", 
    customers, 
    lowBound=0, 
    upBound=capacity, 
    cat='Continuous'
)

# --- OBJECTIVE ---
model += pulp.lpSum(
    c[i][j] * x[r, i, j] 
    for r in vehicles 
    for i in nodes 
    for j in nodes 
    if i != j
), "Minimize_Total_Cost"

# --- CONSTRAINTS ---

# (2) Each customer visited exactly once (Sum of fractions = 1.0)
for j in customers:
    model += pulp.lpSum(x[r, i, j] for r in vehicles for i in nodes if i != j) == 1, f"Visit_{j}"
    
# (3) Each vehicle leaves depot exactly once
for r in vehicles:
    model += pulp.lpSum(x[r, 0, j] for j in customers) == 1, f"Leave_Depot_{r}"

# (4) Flow Conservation
for j in nodes: 
    for r in vehicles:
        inflow = pulp.lpSum(x[r, i, j] for i in nodes if i != j)
        outflow = pulp.lpSum(x[r, j, k] for k in nodes if j != k)
        model += inflow == outflow, f"Flow_Balance_{j}_{r}"

# (5) Capacity Constraint
for r in vehicles:
    model += pulp.lpSum(demands[j] * x[r, i, j] for i in nodes for j in customers if i != j) <= capacity, f"Cap_{r}"

# (6) MTZ Subtour Elimination (Based on provided Image)
# ------------------------------------------------------------------
# The image defines the constraint (1) as: 
# u_j - u_i >= q_j - Q * (1 - x_ijk)
#
# Logic from image:
# IF vehicle r drives i -> j (x=1): 
#    Constraint becomes: u_j >= u_i + q_j 
#    This ensures u_j is at least q_j (demand at j) more than u_i.
#
# IF vehicle r does NOT drive i -> j (x=0):
#    Constraint becomes: u_j - q_j >= u_i - Q
#    This is always valid because u_j >= q_j and u_i <= Q.
# ------------------------------------------------------------------

for r in vehicles:
    for i in customers:
        for j in customers:
            if i != j:
                # We implement: u_j - u_i >= q_j - Q * (1 - x_ijk)
                # Note: We use the parentheses (1 - x[...]) for clarity matching the image
                model += u[j] - u[i] >= demands[j] - capacity * (1 - x[r, i, j]), f"MTZ_{r}_{i}_{j}"

# Constraint (2) from Image: q_i <= u_i <= Q
# The image states: "q_j is the lowest possible value of u_j and Q is the greatest"
for i in customers:
    model += u[i] >= demands[i]  # Lower bound (q_i)
    model += u[i] <= capacity    # Upper bound (Q)

# ==========================================
# SOLVE (LP RELAXATION)
# ==========================================
# Since this is LP, it solves extremely fast compared to MIP
solver = pulp.CPLEX_CMD(timeLimit=60, msg=True) 

try:
    model.solve(solver)
except Exception as e:
    print(f"CPLEX not found, using default: {e}")
    model.solve()

# ==========================================
# 4. RESULTS
# ==========================================
print(f"\nStatus: {pulp.LpStatus[model.status]}")
print(f"LOWER BOUND (LP Relaxation Cost): {pulp.value(model.objective)}")

# Visualizing Fractional Flows
print("\nSignificant Fractional Flows (> 0):")
print("(Format: Vehicle | From -> To | Flow Amount)")
print("-" * 40)

count = 0
for r in vehicles:
    for i in nodes:
        for j in nodes:
            if i != j:
                val = pulp.value(x[r, i, j])
                # Check for non-zero flow (values like 0.5, 0.33 etc)
                if val and val > 0: 
                    print(f"Veh {r} | {i:2d} -> {j:2d} | {val:.2f}")
                    count += 1
                    if count > 20: # Limit output to avoid spamming
                        print("... (too many fractional edges to list)")
                        break
    if count > 20: break


Status: Optimal
LOWER BOUND (LP Relaxation Cost): 625.1446423236698

Significant Fractional Flows (> 0):
(Format: Vehicle | From -> To | Flow Amount)
----------------------------------------
Veh 0 |  0 -> 39 | 0.44
Veh 0 |  0 -> 51 | 0.56
Veh 0 |  3 ->  5 | 0.17
Veh 0 |  5 ->  3 | 0.17
Veh 0 |  6 -> 20 | 0.20
Veh 0 |  7 -> 12 | 0.13
Veh 0 | 10 -> 26 | 0.11
Veh 0 | 12 ->  7 | 0.13
Veh 0 | 13 -> 52 | 0.75
Veh 0 | 15 -> 19 | 0.03
Veh 0 | 16 -> 32 | 0.45
Veh 0 | 17 -> 41 | 0.23
Veh 0 | 18 -> 40 | 0.08
Veh 0 | 19 -> 15 | 0.03
Veh 0 | 20 ->  6 | 0.20
Veh 0 | 22 -> 28 | 0.10
Veh 0 | 23 -> 43 | 0.21
Veh 0 | 26 -> 10 | 0.11
Veh 0 | 28 -> 22 | 0.10
Veh 0 | 29 -> 49 | 0.03
Veh 0 | 30 -> 44 | 0.09
... (too many fractional edges to list)
Veh 0 | 32 -> 16 | 0.45
... (too many fractional edges to list)
Veh 0 | 34 -> 13 | 0.75
... (too many fractional edges to list)
Veh 0 | 35 -> 38 | 0.23
... (too many fractional edges to list)
Veh 0 | 36 -> 50 | 0.03
... (too many fractional edges to list)
Veh 0 | 